# CheXpert Auxiliary Loss for LLaVA Radiology Report Generation

**Dataset:** MIMIC-CXR (simhadrisadaram/mimic-cxr-dataset, study-level pairs + CheXpert-14 labels)  
**Model:** LLaVA-1.5-7B · **Quantisation:** 4-bit NF4 · **Adapters:** QLoRA rank-16  
**Novel Contribution:** CheXpert-14 auxiliary classification head for pathology-aware training

---

### What's New (Novel Contributions)

This notebook extends the baseline QLoRA fine-tuning with a **single novel component**:

| Component | Description | Replaces |
|-----------|-------------|----------|
| **CheXpert Auxiliary Head** | A `Linear(1024, 14)` head on the CLIP CLS token, trained jointly with BCE loss | CXR-LLaVA's 2-stage vision encoder pretraining (Stage 1A + 1B) |

The combined loss is: `Total = CrossEntropy(report) + λ × BCE(pathology_labels)`

This provides pathology-guided supervision during QLoRA fine-tuning **at zero extra data cost** — the CheXpert-14 labels already exist in MIMIC-CXR.

---

### Prerequisites

Run the **updated** `preprocess_local.py` which now includes CheXpert-14 labels.  
The zip file should contain:
- `images/` — 336×336 JPEG chest X-rays, one per study
- `train.json` — list of `{subject_id, study_id, image, report, chexpert}` dicts
- `val.json` — same format, held-out split

The `chexpert` field is a list of 14 binary values (0 or 1) for each pathology.

---

### Optimisation Stack

| Technique | Purpose |
|-----------|--------|
| 4-bit NF4 + double quantisation | Model footprint ~14 GB → ~4 GB |
| QLoRA rank-16 | Only ~0.5% of parameters trained |
| Paged AdamW 8-bit | Optimizer states page to CPU; prevents OOM spikes |
| Gradient checkpointing | Recomputes activations on backward pass; saves peak activation memory |
| NEFTune noise (α=5) | Embedding regularisation; improves generation quality |
| Group-by-length batching | Minimises per-batch padding tokens |
| Cosine LR with 3% warmup | Stable convergence from cold start |
| Frozen vision tower | CLIP ViT already generalises well; updating it wastes capacity |
| **[NOVEL] CheXpert Aux Head** | Pathology classification guides visual representations |

---
## Step 1 — Global Configuration

All hyperparameters are centralised here. Adjust paths and training knobs in one place before running any other cell.

**[NOVEL]** `AUX_LOSS_LAMBDA` controls the weight of the CheXpert classification loss.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ZIP_PATH       = "/content/drive/MyDrive/preprocessed_cxr.zip"   # adjust if needed
DATA_DIR       = "/content/cxr_data"
CHECKPOINT_DIR = "/content/drive/MyDrive/llava_cxr_aux_checkpoints"
LORA_SAVE_DIR  = "/content/drive/MyDrive/llava_cxr_aux_lora_final"

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID       = "llava-hf/llava-1.5-7b-hf"

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R         = 16     # rank; higher = more capacity, more memory
LORA_ALPHA     = 32     # scaling = alpha / r; 2× rank is a stable default
LORA_DROPOUT   = 0.05

# ── Training ───────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH      = 512   # covers > 95% of report lengths in this dataset
PER_DEVICE_BATCH    = 2     # physical micro-batch per GPU
GRAD_ACCUM_STEPS    = 8     # effective batch size = 2 × 8 = 16
LEARNING_RATE       = 2e-4
WARMUP_RATIO        = 0.03
MAX_TRAIN_SAMPLES   = None  # set e.g. 500 for a quick smoke-test

# ── [NOVEL] Auxiliary Loss ─────────────────────────────────────────────────
AUX_LOSS_LAMBDA     = 0.3   # weight of CheXpert classification loss
NUM_CHEXPERT_LABELS = 14    # Atelectasis, Cardiomegaly, ..., Support Devices

# ── CheXpert-14 Label Names (for reference) ───────────────────────────────
CHEXPERT_LABELS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices"
]

print("Configuration loaded.")
print(f"  Auxiliary loss weight (λ): {AUX_LOSS_LAMBDA}")

---
## Step 2 — Install Dependencies

Pins to the latest stable releases. Additional packages for quantitative evaluation: `nltk`, `rouge-score`, `scikit-learn`.

In [ ]:
!pip install -q --upgrade transformers accelerate peft bitsandbytes pillow tqdm
!pip install -q nltk rouge-score scikit-learn

---
## Step 3 — Mount Drive and Extract Data

The archive is extracted once into `/content/cxr_data`. This cell verifies that the `chexpert` field exists in the dataset.

In [ ]:
import os, json, zipfile, torch
from google.colab import drive

drive.mount("/content/drive")

if not os.path.exists(f"{DATA_DIR}/train.json"):
    print("Extracting archive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)
    print("Done.")
else:
    print("Data already extracted — skipping.")

train_data = json.load(open(f"{DATA_DIR}/train.json"))
val_data   = json.load(open(f"{DATA_DIR}/val.json"))

print(f"Training samples : {len(train_data):,}")
print(f"Validation samples: {len(val_data):,}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — check runtime'}")

# Inspect one sample to verify preprocessing output is correct
s = train_data[0]
print(f"\nSample record:")
print(f"  subject_id : {s['subject_id']}")
print(f"  study_id   : {s['study_id']}")
print(f"  image path : {s['image']}")
print(f"  report[:80]: {s['report'][:80]}")

# [NOVEL] Verify CheXpert labels exist
if 'chexpert' in s:
    print(f"  chexpert   : {s['chexpert']}")
    print(f"  positive findings: {sum(s['chexpert'])} / 14")
else:
    print("\n⚠️  WARNING: 'chexpert' field not found in dataset!")
    print("    Run the updated preprocess_local.py to include CheXpert labels.")
    print("    Falling back to zero labels (aux loss will not provide signal).")

---
## Step 4 — Load LLaVA-1.5-7B with 4-bit Quantisation

**NF4 (NormalFloat-4)** is an information-theoretically optimal 4-bit data type for weights that follow a normal distribution.

**Double quantisation** compresses the quantisation scaling constants themselves into 8-bit.

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.float16,
    low_cpu_mem_usage   = True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Parameters : {sum(p.numel() for p in model.parameters())/1e9:.2f} B")
print(f"VRAM used  : {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Step 5 — Inject QLoRA Adapters

LoRA decomposes each weight update into two low-rank matrices. All seven projection matrices in each transformer block are targeted.

The vision tower (CLIP ViT-L/14) is explicitly frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

# Freeze vision tower
for name, param in model.named_parameters():
    if "vision_tower" in name:
        param.requires_grad = False

model.print_trainable_parameters()
print(f"VRAM after adapter injection: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Step 6 — [NOVEL] CheXpert Auxiliary Head and Forward Hook

**This is the novel contribution.**

We add a small classification head (`Linear(1024, 14)`) that predicts the presence of 14 pathologies from the CLIP CLS token. A forward hook captures the CLS token during the model's forward pass at zero extra compute cost.

**Why this works:**
- The CLS token is a global summary of the entire image (aggregated via self-attention over all 576 patches)
- Training the aux head with BCE loss against CheXpert labels teaches the model to extract pathology-relevant features
- The gradient flows back through the projection layer to influence the LLM's understanding of visual tokens

**Architecture:**
```
CLIP output: [B, 577, 1024]
                ↓ position 0
CLS token:   [B, 1024]
                ↓ Linear(1024, 14)
Aux logits:  [B, 14]  →  BCEWithLogitsLoss vs CheXpert labels
```

In [ ]:
import torch.nn as nn

# ═══════════════════════════════════════════════════════════════════════════
# [NOVEL] CheXpert Auxiliary Classification Head
# ═══════════════════════════════════════════════════════════════════════════

class CheXpertAuxHead(nn.Module):
    """
    Lightweight classification head for 14 CheXpert pathologies.
    
    Input:  CLS token from CLIP vision tower, shape [B, 1024]
    Output: Logits for 14 pathologies, shape [B, 14]
    
    Total parameters: 1024 * 14 + 14 = 14,350 (~0.01 MB)
    """
    def __init__(self, in_dim=1024, num_labels=14):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_labels)
    
    def forward(self, cls_token):
        return self.fc(cls_token)


# Create and move to GPU
aux_head = CheXpertAuxHead(in_dim=1024, num_labels=NUM_CHEXPERT_LABELS)
aux_head = aux_head.to(model.device).half()  # FP16 to match model

print(f"Aux head parameters: {sum(p.numel() for p in aux_head.parameters()):,}")
print(f"Aux head size: {sum(p.numel() * p.element_size() for p in aux_head.parameters()) / 1e6:.4f} MB")

# ═══════════════════════════════════════════════════════════════════════════
# [NOVEL] Forward Hook to Capture CLS Token
# ═══════════════════════════════════════════════════════════════════════════

_captured_cls = {}

def _cls_hook(module, input, output):
    """
    Hook that captures the CLS token (position 0) from the vision tower output.
    
    This runs during the model's forward pass — no extra forward call needed.
    The CLS token is stored in _captured_cls['cls'] for use in loss computation.
    
    Args:
        output: Tuple where output[0] is last_hidden_state of shape [B, 577, 1024]
                Position 0 is the CLS token, positions 1-576 are patch tokens.
    """
    # output[0] shape: [B, 577, 1024]
    # We extract position 0 (CLS token) → [B, 1024]
    _captured_cls["cls"] = output[0][:, 0, :].detach().clone()


# Register hook on the vision tower's encoder
# The exact module path depends on LLaVA's architecture
vision_encoder = model.base_model.model.vision_tower.vision_model.encoder
hook_handle = vision_encoder.register_forward_hook(_cls_hook)

print("\n✓ CheXpert auxiliary head created")
print("✓ Forward hook registered on vision encoder")
print(f"\nVRAM after aux head: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Step 7 — [NOVEL] Dataset with CheXpert Labels

**Modified from baseline:** Each sample now includes `chexpert_labels` — a tensor of 14 binary values indicating pathology presence.

The dataset gracefully handles missing labels by defaulting to zeros (no pathology signal).

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset

PROMPT = (
    "USER: <image>\n"
    "Analyze this chest X-ray and write a detailed radiology report "
    "including findings and impression.\n"
    "ASSISTANT: "
)

class ChestXrayDatasetWithCheXpert(Dataset):
    """
    [NOVEL] Extended dataset that includes CheXpert-14 pathology labels.
    
    Returns:
        input_ids:       [MAX_SEQ_LENGTH] - tokenized prompt + report
        attention_mask:  [MAX_SEQ_LENGTH] - 1 for real tokens, 0 for padding
        pixel_values:    [3, 336, 336]    - normalized image
        labels:          [MAX_SEQ_LENGTH] - target token IDs with prompt masked to -100
        chexpert_labels: [14]             - binary pathology labels (NOVEL)
    """
    def __init__(self, records, proc, data_dir, max_len):
        self.records    = records
        self.proc       = proc
        self.data_dir   = data_dir
        self.max_len    = max_len
        self.prompt_len = len(proc.tokenizer(PROMPT, add_special_tokens=True)["input_ids"])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        image = Image.open(os.path.join(self.data_dir, rec["image"])).convert("RGB")
        text  = PROMPT + rec["report"].strip() + self.proc.tokenizer.eos_token

        enc = self.proc(
            text           = text,
            images         = image,
            return_tensors = "pt",
            padding        = "max_length",
            max_length     = self.max_len,
            truncation     = True,
        )

        ids    = enc["input_ids"].squeeze(0)
        labels = ids.clone()
        labels[: self.prompt_len] = -100
        pad_id = self.proc.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        # ═══════════════════════════════════════════════════════════════════
        # [NOVEL] Extract CheXpert labels
        # ═══════════════════════════════════════════════════════════════════
        # Default to zeros if 'chexpert' field is missing
        chexpert_raw = rec.get("chexpert", [0.0] * NUM_CHEXPERT_LABELS)
        chexpert_labels = torch.tensor(chexpert_raw, dtype=torch.float16)

        return {
            "input_ids":       ids,
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "pixel_values":    enc["pixel_values"].squeeze(0),
            "labels":          labels,
            "chexpert_labels": chexpert_labels,  # [NOVEL]
        }


class StackCollator:
    """Collates pre-padded batches by stacking — no extra padding pass."""
    def __call__(self, features):
        return {k: torch.stack([f[k] for f in features]) for k in features[0]}


train_recs    = train_data[:MAX_TRAIN_SAMPLES] if MAX_TRAIN_SAMPLES else train_data
train_dataset = ChestXrayDatasetWithCheXpert(train_recs, processor, DATA_DIR, MAX_SEQ_LENGTH)
val_dataset   = ChestXrayDatasetWithCheXpert(val_data,   processor, DATA_DIR, MAX_SEQ_LENGTH)

print(f"Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}")

# Verify output shapes
sample = train_dataset[0]
print(f"\nSample tensors:")
print(f"  input_ids      : {sample['input_ids'].shape}")
print(f"  pixel_values   : {sample['pixel_values'].shape}")
print(f"  active labels  : {(sample['labels'] != -100).sum().item()} tokens")
print(f"  chexpert_labels: {sample['chexpert_labels'].shape} → {sample['chexpert_labels'].tolist()}")
print(f"  positive pathologies: {sample['chexpert_labels'].sum().item():.0f} / 14")

---
## Step 8 — [NOVEL] Custom Trainer with Combined Loss

**This is the core novel implementation.**

We subclass HuggingFace's `Trainer` to override `compute_loss()`. The combined loss is:

```
Total Loss = CrossEntropy(generated_report, ground_truth_report)
           + λ × BCEWithLogitsLoss(aux_logits, chexpert_labels)
```

**Loss components:**
- **Generation loss (CE):** Trains the LLM to produce correct report tokens
- **Auxiliary loss (BCE):** Trains the model to recognize pathologies from visual features

**Why this helps:**
The auxiliary loss provides an additional gradient signal that encourages the vision-language alignment to be pathology-aware, without requiring a separate pretraining stage.

In [ ]:
from transformers import TrainingArguments, Trainer
import torch.nn as nn

# ═══════════════════════════════════════════════════════════════════════════
# [NOVEL] Custom Trainer with CheXpert Auxiliary Loss
# ═══════════════════════════════════════════════════════════════════════════

class CXRTrainerWithAuxLoss(Trainer):
    """
    Custom Trainer that adds CheXpert auxiliary classification loss.
    
    Combined loss: gen_loss + λ × aux_loss
    
    The aux_loss is BCEWithLogitsLoss between:
      - Predictions from aux_head(CLS_token)
      - Ground truth CheXpert-14 binary labels
    
    Important: This trainer includes aux_head parameters in the optimizer
    so they get updated during training.
    """
    
    def __init__(self, *args, aux_head, aux_lambda, captured_cls_dict, **kwargs):
        super().__init__(*args, **kwargs)
        self.aux_head = aux_head
        self.aux_lambda = aux_lambda
        self.captured_cls_dict = captured_cls_dict
        self.bce_loss = nn.BCEWithLogitsLoss()
        
        # For logging
        self._last_gen_loss = 0.0
        self._last_aux_loss = 0.0
    
    def create_optimizer(self):
        """
        [NOVEL] Override to include aux_head parameters in the optimizer.
        
        By default, Trainer only optimizes model.parameters(). We also
        include aux_head.parameters() so the classification head learns.
        """
        if self.optimizer is None:
            # Get trainable model parameters (LoRA adapters)
            model_params = [p for p in self.model.parameters() if p.requires_grad]
            
            # Add aux_head parameters
            aux_params = list(self.aux_head.parameters())
            
            # Combine all trainable parameters
            all_params = model_params + aux_params
            
            # Create optimizer with all parameters
            optimizer_cls, optimizer_kwargs = Trainer.get_optimizer_cls_and_kwargs(self.args)
            self.optimizer = optimizer_cls(all_params, **optimizer_kwargs)
            
            print(f"Optimizer created:")
            print(f"  Model params (LoRA): {sum(p.numel() for p in model_params):,}")
            print(f"  Aux head params: {sum(p.numel() for p in aux_params):,}")
        
        return self.optimizer
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Compute combined generation + auxiliary loss.
        
        Steps:
        1. Pop chexpert_labels from inputs (not needed by model.forward)
        2. Run model forward (this triggers the hook, capturing CLS token)
        3. Get generation loss from model output
        4. Compute aux loss from aux_head(CLS_token) vs chexpert_labels
        5. Return combined loss
        """
        # Extract CheXpert labels (not used by model.forward)
        chexpert_labels = inputs.pop("chexpert_labels")
        
        # Forward pass — this triggers the hook and captures CLS token
        outputs = model(**inputs)
        gen_loss = outputs.loss
        
        # Get captured CLS token from hook
        cls_token = self.captured_cls_dict.get("cls")
        
        if cls_token is not None and self.aux_lambda > 0:
            # Compute auxiliary classification loss
            # cls_token: [B, 1024], enable gradients for backward through aux_head
            cls_for_aux = cls_token.clone().requires_grad_(True)
            aux_logits = self.aux_head(cls_for_aux.float())  # [B, 14]
            aux_loss = self.bce_loss(aux_logits, chexpert_labels.float())
            
            # Combined loss
            total_loss = gen_loss + self.aux_lambda * aux_loss
            
            # Store for logging
            self._last_gen_loss = gen_loss.item()
            self._last_aux_loss = aux_loss.item()
        else:
            # Fallback: no aux loss (CLS not captured or lambda=0)
            total_loss = gen_loss
            self._last_gen_loss = gen_loss.item()
            self._last_aux_loss = 0.0
        
        return (total_loss, outputs) if return_outputs else total_loss


print("CXRTrainerWithAuxLoss defined.")
print(f"  Auxiliary loss weight (λ): {AUX_LOSS_LAMBDA}")

---
## Step 9 — Training

Training uses the custom `CXRTrainerWithAuxLoss` which computes the combined loss.

**Expected behavior:**
- `gen_loss` should decrease from ~4.0 to ~1.0 over one epoch
- `aux_loss` should decrease from ~0.693 (random) to ~0.25-0.35 (learned)

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir                    = CHECKPOINT_DIR,
    num_train_epochs              = 1,

    # Batch size and accumulation
    per_device_train_batch_size   = PER_DEVICE_BATCH,
    per_device_eval_batch_size    = PER_DEVICE_BATCH,
    gradient_accumulation_steps   = GRAD_ACCUM_STEPS,

    # Optimizer and schedule
    optim                         = "paged_adamw_8bit",
    learning_rate                 = LEARNING_RATE,
    weight_decay                  = 0.01,
    warmup_ratio                  = WARMUP_RATIO,
    lr_scheduler_type             = "cosine",
    max_grad_norm                 = 1.0,

    # Memory and precision
    fp16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},

    # Quality improvements
    neftune_noise_alpha           = 5,
    group_by_length               = True,

    # Logging and checkpointing
    logging_steps                 = 25,
    eval_strategy                 = "steps",
    eval_steps                    = 1000,
    save_steps                    = 500,
    save_total_limit              = 3,
    save_safetensors              = True,

    # Data loading
    dataloader_num_workers        = 2,
    dataloader_pin_memory         = True,
    dataloader_persistent_workers = True,

    remove_unused_columns         = False,
    report_to                     = "none",
    seed                          = 42,
)

# Auto-detect latest checkpoint for seamless resume
ckpts = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
) if os.path.exists(CHECKPOINT_DIR) else []
resume_ckpt = os.path.join(CHECKPOINT_DIR, ckpts[-1]) if ckpts else None
print("Resuming from:", resume_ckpt or "scratch")

# ═══════════════════════════════════════════════════════════════════════════
# [NOVEL] Use custom trainer with auxiliary loss
# ═══════════════════════════════════════════════════════════════════════════
trainer = CXRTrainerWithAuxLoss(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    data_collator   = StackCollator(),
    aux_head        = aux_head,           # [NOVEL]
    aux_lambda      = AUX_LOSS_LAMBDA,    # [NOVEL]
    captured_cls_dict = _captured_cls,    # [NOVEL]
)

print(f"\nStarting training with auxiliary loss (λ={AUX_LOSS_LAMBDA})...")
trainer.train(resume_from_checkpoint=resume_ckpt)
print("Training complete.")

---
## Step 10 — Save LoRA Adapters and Auxiliary Head

**[NOVEL]** We save both the LoRA adapters AND the auxiliary head weights.

In [ ]:
os.makedirs(LORA_SAVE_DIR, exist_ok=True)

# Save LoRA adapters
model.save_pretrained(LORA_SAVE_DIR)
processor.save_pretrained(LORA_SAVE_DIR)

# [NOVEL] Save auxiliary head
aux_head_path = os.path.join(LORA_SAVE_DIR, "aux_head.pt")
torch.save(aux_head.state_dict(), aux_head_path)

size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk(LORA_SAVE_DIR) for f in fns
) / 1e6

print(f"Saved to : {LORA_SAVE_DIR}")
print(f"Size     : {size_mb:.1f} MB")
print(f"Aux head : {aux_head_path}")

---
## Step 11 — Qualitative Inference

Generate reports for a few validation samples to verify quality.

In [ ]:
from PIL import Image

model.eval()

INFER_PROMPT = (
    "USER: <image>\n"
    "Analyze this chest X-ray and write a detailed radiology report "
    "including findings and impression.\n"
    "ASSISTANT: "
)

for i, rec in enumerate(val_data[:3]):
    img = Image.open(os.path.join(DATA_DIR, rec["image"])).convert("RGB")
    inp = processor(text=INFER_PROMPT, images=img, return_tensors="pt")
    inp = {k: v.to(model.device) for k, v in inp.items()}

    with torch.no_grad():
        out_ids = model.generate(**inp, max_new_tokens=300, do_sample=False)

    generated = processor.tokenizer.decode(out_ids[0], skip_special_tokens=True)
    response  = generated.split("ASSISTANT:")[-1].strip()

    print(f"\n{'─'*60}  Sample {i+1}  (subject={rec['subject_id']} study={rec['study_id']})")
    
    # Show CheXpert labels if available
    if 'chexpert' in rec:
        positive = [CHEXPERT_LABELS[j] for j, v in enumerate(rec['chexpert']) if v == 1]
        print(f"CHEXPERT LABELS: {positive if positive else ['No Finding']}")
    
    print(f"\nGENERATED:\n{response}")
    print(f"\nGROUND TRUTH:\n{rec['report'][:400]}")

---
## Step 12 — [NOVEL] Quantitative Evaluation

Compute standard metrics for radiology report generation:
- **BLEU-4:** N-gram precision (standard NLP metric)
- **ROUGE-L:** Longest common subsequence F1
- **Per-pathology AUC:** Classification performance of the auxiliary head

For full publication, also compute **CheXbert F1** using the `f1chexbert` package.

In [ ]:
import nltk
from rouge_score import rouge_scorer
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ═══════════════════════════════════════════════════════════════════════════
# [NOVEL] Comprehensive Evaluation
# ═══════════════════════════════════════════════════════════════════════════

def compute_bleu4(reference, hypothesis):
    """Compute BLEU-4 score."""
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    ref_tokens = nltk.word_tokenize(reference.lower())
    hyp_tokens = nltk.word_tokenize(hypothesis.lower())
    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref_tokens], hyp_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)


def evaluate_model(model, processor, val_data, data_dir, aux_head, num_samples=100):
    """
    Evaluate model on validation set.
    
    Returns:
        dict with BLEU-4, ROUGE-L, and per-pathology AUC scores
    """
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    bleu_scores = []
    rouge_scores = []
    all_aux_preds = []
    all_aux_labels = []
    
    samples = val_data[:num_samples]
    
    for rec in tqdm(samples, desc="Evaluating"):
        img = Image.open(os.path.join(data_dir, rec["image"])).convert("RGB")
        inp = processor(text=INFER_PROMPT, images=img, return_tensors="pt")
        inp = {k: v.to(model.device) for k, v in inp.items()}
        
        with torch.no_grad():
            out_ids = model.generate(**inp, max_new_tokens=300, do_sample=False)
        
        generated = processor.tokenizer.decode(out_ids[0], skip_special_tokens=True)
        response = generated.split("ASSISTANT:")[-1].strip()
        reference = rec["report"]
        
        # BLEU-4
        bleu = compute_bleu4(reference, response)
        bleu_scores.append(bleu)
        
        # ROUGE-L
        rouge = scorer.score(reference, response)['rougeL'].fmeasure
        rouge_scores.append(rouge)
        
        # Auxiliary predictions (if labels available)
        if 'chexpert' in rec and _captured_cls.get("cls") is not None:
            cls_token = _captured_cls["cls"]
            with torch.no_grad():
                aux_logits = aux_head(cls_token.float())
                aux_probs = torch.sigmoid(aux_logits).cpu().numpy()
            all_aux_preds.append(aux_probs[0])
            all_aux_labels.append(rec['chexpert'])
    
    results = {
        "BLEU-4": np.mean(bleu_scores),
        "ROUGE-L": np.mean(rouge_scores),
        "num_samples": len(samples),
    }
    
    # Per-pathology AUC
    if all_aux_preds:
        all_aux_preds = np.array(all_aux_preds)
        all_aux_labels = np.array(all_aux_labels)
        
        auc_scores = {}
        for j, label_name in enumerate(CHEXPERT_LABELS):
            try:
                if len(np.unique(all_aux_labels[:, j])) > 1:  # Need both classes
                    auc = roc_auc_score(all_aux_labels[:, j], all_aux_preds[:, j])
                    auc_scores[label_name] = auc
            except:
                pass
        
        results["AUC_per_pathology"] = auc_scores
        results["AUC_mean"] = np.mean(list(auc_scores.values())) if auc_scores else 0.0
    
    return results


# Run evaluation
print("Running quantitative evaluation...")
print("(Set num_samples higher for final results)\n")

results = evaluate_model(
    model, processor, val_data, DATA_DIR, aux_head,
    num_samples=50  # Increase for final evaluation
)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Samples evaluated: {results['num_samples']}")
print(f"\nReport Generation:")
print(f"  BLEU-4  : {results['BLEU-4']:.4f}")
print(f"  ROUGE-L : {results['ROUGE-L']:.4f}")

if 'AUC_mean' in results:
    print(f"\nPathology Classification (Auxiliary Head):")
    print(f"  Mean AUC: {results['AUC_mean']:.4f}")
    print(f"\n  Per-pathology AUC:")
    for name, auc in results.get('AUC_per_pathology', {}).items():
        print(f"    {name:30s}: {auc:.4f}")

---
## Step 13 — Load Saved Adapters (New Session)

To restore the fine-tuned model in a future session without retraining:
1. Run Steps 1–5 (configuration, install, mount, base model load, QLoRA prep)
2. Run Step 6 to create the aux head structure
3. Run this cell to load saved weights

In [ ]:
from peft import PeftModel

# Load LoRA adapters
model = PeftModel.from_pretrained(model, LORA_SAVE_DIR)
model.eval()

# [NOVEL] Load auxiliary head
aux_head_path = os.path.join(LORA_SAVE_DIR, "aux_head.pt")
if os.path.exists(aux_head_path):
    aux_head.load_state_dict(torch.load(aux_head_path))
    aux_head.eval()
    print(f"Aux head loaded from: {aux_head_path}")

print(f"Adapters loaded from: {LORA_SAVE_DIR}")
print("Proceed to Step 11 for inference or Step 12 for evaluation.")